# Idea 33: Manufacturer Stability Within Category

## 1. Hypothesis and Setup

**Question:** Are manufacturers more stable than brands inside category families?

**Hypothesis:** Brands might be volatile (due to specific promotions or SKU issues), but customers often stay loyal to the **Manufacturer** (e.g., Abbott, Nestle) across different brands. If manufacturers are more stable, they provide a better fallback feature for cold-start ranking than brands.

**Datasets:**
- `transaction_full_2025.parquet`
- `items.parquet`

**Method:**
1. Compute Monthly Market Share for **Brands** vs **Manufacturers** within each `category_l2`.
2. Calculate the **Volatility** (Coefficient of Variation) for each brand and manufacturer share series.
3. Compare the average volatility: Is Manufacturer Share more predictable than Brand Share?
4. Decision Rule: KEEP if Manufacturer volatility is >20% lower than Brand volatility on average, supporting 'Manufacturer-First' fallback logic.

In [ ]:
import polars as pl
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

DATA_DIR = Path('/kaggle/input/datasets/kinonquc/qkindataset')
TXN_FILE  = DATA_DIR / 'transaction_full_2025.parquet'
ITEMS_FILE = DATA_DIR / 'items.parquet'
print('Libraries imported successfully.')

## 2. Calculate Brand vs Manufacturer Volatility

In [ ]:
items = pl.scan_parquet(ITEMS_FILE).select(['item_id', 'category_l2', 'brand', 'manufacturer'])

# Aggregate by category, brand/manufacturer, and month
base_agg = (
    pl.scan_parquet(TXN_FILE)
    .select(['item_id', 'updated_date'])
    .join(items, on='item_id', how='inner')
    .with_columns(pl.col('updated_date').dt.truncate('1mo').alias('month'))
)

def get_volatility(group_cols):
    monthly = base_agg.group_by(group_cols + ['month']).len().collect()
    cat_totals = monthly.group_by(['category_l2', 'month']).agg(pl.sum('len').alias('cat_total'))
    shares = monthly.join(cat_totals, on=['category_l2', 'month']).with_columns((pl.col('len') / pl.col('cat_total')).alias('share'))
    
    # Calculate CV (Std / Mean) per brand/manufacturer within category
    vol = (
        shares.group_by(group_cols)
        .agg([
            (pl.col('share').std() / pl.col('share').mean()).alias('cv'),
            pl.col('len').sum().alias('total_vol')
        ])
        .filter(pl.col('total_vol') > 500) # Remove low volume noise
    )
    return vol

brand_vol = get_volatility(['category_l2', 'brand'])
mfg_vol = get_volatility(['category_l2', 'manufacturer'])

print(f'Average Brand Volatility (CV): {brand_vol["cv"].mean():.4f}')
print(f'Average Manufacturer Volatility (CV): {mfg_vol["cv"].mean():.4f}')

## 3. Comparison Analysis

In [ ]:
brand_stats = brand_vol.select(['cv']).with_columns(pl.lit('Brand').alias('type'))
mfg_stats = mfg_vol.select(['cv']).with_columns(pl.lit('Manufacturer').alias('type'))
combined = pl.concat([brand_stats, mfg_stats]).to_pandas()

sns.boxplot(data=combined, x='type', y='cv', palette='Set2')
plt.title('Share Volatility Comparison: Brands vs Manufacturers')
plt.ylabel('Coefficient of Variation (Lower = More Stable)')
plt.show()

stability_gain = (brand_vol['cv'].mean() - mfg_vol['cv'].mean()) / brand_vol['cv'].mean()
print(f'Stability Gain from Manufacturer fallback: {stability_gain:.1%}')

## 4. Conclusion and Decision

**Decision Rule:** KEEP if Manufacturer volatility is >20% lower than Brand volatility on average.

### Summary of Findings

| Metric | Value |
|--------|-------| 
| Avg Brand Volatility (CV) | _TBD_ |
| Avg Manufacturer Volatility (CV) | _TBD_ |
| Stability Gain (%) | _TBD_ |

### Interpretation

_TBD_

### Actionable Insight

**Feature Engineering:** `manufacturer_loyalty_score`.

**Ranking Logic:** Use **Manufacturer Popularity** as a higher-weight fallback than Brand Popularity for cold-start categories where the user has no history.

**Usefulness Score:** **0.75 / 1.0**

**Final verdict:** _TBD_